# **Redes Neuronales Convolucionales**
### Actividad grupal

---

En esta actividad se trabajará con Redes Neuronales Convolucionales (Convolutional Neural Networks, CNN) para resolver un problema de clasificación de imágenes. En particular, se utilizará un conjunto de imágenes de mascotas, enfocándose en la clasificación de perros y gatos.

Dado que las CNN profundas son modelos computacionalmente demandantes, se recomienda realizar la práctica en Google Colaboratory, aprovechando el soporte para unidades de procesamiento gráfico (GPU). En el siguiente enlace se describe el procedimiento para habilitar una GPU en Colab:
[Guía para activar GPU en Google Colab](https://medium.com/deep-learning-turkey/google-colab-free-gpu-tutorial-e113627b9f5d).

El conjunto de datos a utilizar es el **Oxford-IIIT Pet Dataset**, el cual contiene imágenes de distintas razas de perros y gatos. Su sitio oficial es:
[Oxford-IIIT Pet Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/)

Este dataset es considerablemente más complejo que otros utilizados previamente, como Fashion MNIST, ya que:
- Incluye múltiples clases (razas),
- Presenta variaciones significativas en escala, pose e iluminación,
- Contiene imágenes con diferentes fondos y encuadres.

El conjunto de datos puede integrarse fácilmente a flujos de trabajo basados en frameworks de aprendizaje profundo como [Tensorflow](https://www.tensorflow.org/datasets/catalog/oxford_iiit_pet) y [PyTorch](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.OxfordIIITPet.html), ya sea mediante utilidades propias de cada framework o mediante descarga directa desde el sitio oficial.

## Enunciado del ejercicio

Utilizando ***Convolutional Neural Networks*** (CNN) implementadas con Keras, entrenar un clasificador capaz de reconocer razas de perros y gatos en imágenes, alcanzando una accuracy en el conjunto de **test** de al menos **85%**. Es indispensable que el **mejor modelo seleccionado no presente evidencias de overfitting ni de underfitting**.

El informe debe abordar: análisis exploratorio, diseño y evaluación de al menos cuatro modelos (uno propio), análisis de resultados con *precision*/*recall* por clase y matriz de confusión, análisis visual de errores, comparación CNN vs. red *fully connected*, comparación entre arquitecturas CNN y empleo de *data augmentation*.

---
## Índice de la solución

| Sección | Contenido |
|---|---|
| 1 | Configuración del entorno, semillas y reproducibilidad |
| 2 | Carga del dataset y definición de particiones *train / validation / test* |
| 3 | Análisis exploratorio de los datos (EDA) |
| 4 | *Pipeline* `tf.data`, normalización y *data augmentation* |
| 5 | Utilidades de entrenamiento, guardado y evaluación (trazabilidad experimental) |
| 6 | **Modelo 1** — Red *Fully Connected* (baseline con imágenes aplanadas) |
| 7 | **Modelo 2** — CNN propia entrenada desde cero |
| 8 | **Modelo 3** — CNN propia + *data augmentation* + regularización |
| 9 | **Modelo 4** — *Transfer learning*: MobileNetV2 congelada (*feature extraction*) |
| 10 | **Modelo 5** — *Fine-tuning* de MobileNetV2 (**mejor modelo**) |
| 11 | **Modelo 6 (opcional)** — *Fine-tuning* de EfficientNetV2-B0 |
| 12 | Comparativa global de modelos |
| 13 | Diagnóstico de *overfitting* / *underfitting* |
| 14 | Evaluación final sobre **test** del mejor modelo (precision/recall y matriz de confusión) |
| 15 | Análisis visual de errores |
| 16 | Conclusiones e informe técnico |

**Resumen del enfoque.** El problema es de granularidad fina (*fine-grained*): 37 razas con muy pocas imágenes por clase (~100 en entrenamiento). Una CNN entrenada desde cero satura muy por debajo del objetivo, mientras que el *transfer learning* sobre una red preentrenada en ImageNet, seguido de un *fine-tuning* con tasa de aprendizaje muy baja, permite superar el 85% exigido. Todos los experimentos comparten semilla, particiones, *pipeline* de datos y protocolo de evaluación para que la comparación sea justa.

---
## 1. Configuración del entorno

Se fijan semillas globales (`keras.utils.set_random_seed`) y se centralizan todos los hiperparámetros en una única celda para garantizar la **trazabilidad experimental**: cualquier resultado del notebook puede reproducirse conociendo esta configuración.

In [ ]:
# Si alguna librería no está disponible en el entorno, descomentar:
# !pip install -q tensorflow tensorflow-datasets pandas

import json
import os
import platform
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from keras import layers
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

print("Python           :", platform.python_version())
print("TensorFlow       :", tf.__version__)
print("Keras            :", keras.__version__)
print("GPU disponible   :", tf.config.list_physical_devices("GPU"))

In [ ]:
# ---------------------------------------------------------------------------
# Configuración global del experimento (única fuente de verdad)
# ---------------------------------------------------------------------------
SEED = 42
IMG_SIZE = 160                 # las imágenes se estandarizan a 160x160 px
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Épocas máximas por modelo (EarlyStopping decide la parada real)
EPOCHS_FC = 40                 # Modelo 1 - Fully Connected
EPOCHS_CNN = 60                # Modelos 2 y 3 - CNN propia
EPOCHS_TL = 25                 # Modelo 4 - Transfer learning (cabeza)
EPOCHS_FT = 20                 # Modelo 5 - Fine-tuning
FINE_TUNE_AT = 100             # capa de MobileNetV2 a partir de la cual se descongela

RUN_MODEL_6 = True             # Modelo opcional (EfficientNetV2-B0). Poner en False para ahorrar tiempo
FORCE_RETRAIN = False          # True fuerza el reentrenamiento aunque exista checkpoint

MODELS_DIR = Path("models")    # mejores pesos de cada modelo (.keras)
HIST_DIR = Path("histories")   # curvas de entrenamiento (.json)
REPORT_DIR = Path("reports")   # tablas y figuras para el informe
for d in (MODELS_DIR, HIST_DIR, REPORT_DIR):
    d.mkdir(exist_ok=True)

# Reproducibilidad: fija las semillas de Python, NumPy y TensorFlow
keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
# Nota: tf.config.experimental.enable_op_determinism() haría el entrenamiento
# totalmente determinista en GPU, pero penaliza el tiempo de cómputo (~1.5x).
# Se documenta como opción y no se activa por defecto.

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---
## 2. Carga del dataset y particiones

El *split* `train` de `oxford_iiit_pet` en TFDS contiene 3.680 imágenes y el *split* `test` otras 3.669. Siguiendo el guion de la actividad, se parte **únicamente del split `train`**, dividiéndolo en 80% entrenamiento / 10% validación / 10% test.

> **Corrección respecto al enunciado.** La celda original utilizaba `'train[:80]'`, que en la sintaxis de *slicing* de TFDS significa "los **80 primeros ejemplos**" (no el 80%) y además solapa con el resto de particiones. Se corrige a `'train[:80%]'`, de modo que las tres particiones son **disjuntas y exhaustivas**: 80% / 10% / 10%.

El conjunto de **test así construido se reserva exclusivamente para la evaluación final**; no interviene en el entrenamiento ni en la selección de modelos ni de hiperparámetros.

In [ ]:
(train_raw, val_raw, test_raw), info = tfds.load(
    "oxford_iiit_pet",
    split=[
        "train[:80%]",      # entrenamiento (80%)
        "train[80%:90%]",   # validación   (10%)
        "train[90%:]",      # prueba       (10%)
    ],
    with_info=True,
    as_supervised=True,     # devuelve tuplas (imagen, etiqueta)
    shuffle_files=False,    # orden determinista -> particiones reproducibles
)

CLASS_NAMES = info.features["label"].names
N_CLASSES = info.features["label"].num_classes

N_TRAIN = train_raw.cardinality().numpy()
N_VAL = val_raw.cardinality().numpy()
N_TEST = test_raw.cardinality().numpy()

print(f"Número de clases (razas): {N_CLASSES}")
print(f"Entrenamiento : {N_TRAIN} imágenes")
print(f"Validación    : {N_VAL} imágenes")
print(f"Test          : {N_TEST} imágenes")
print(f"Total         : {N_TRAIN + N_VAL + N_TEST} imágenes")
print("\nPrimeras clases:", CLASS_NAMES[:5], "...")

---
## 3. Análisis exploratorio de los datos (EDA)

Se analizan: (i) el número de clases y su distribución, (ii) el balance entre especies (perro/gato), (iii) las dimensiones originales de las imágenes y (iv) ejemplos representativos. Este análisis condiciona decisiones posteriores (tamaño de entrada, necesidad de *data augmentation*, uso de *accuracy* como métrica principal).

In [ ]:
# Una única pasada sobre el split completo para extraer metadatos:
# etiqueta (raza), especie (0=gato, 1=perro) y dimensiones originales.
meta_rows = []
for ex in tfds.load("oxford_iiit_pet", split="train", shuffle_files=False):
    h, w = int(ex["image"].shape[0]), int(ex["image"].shape[1])
    meta_rows.append(
        {
            "label": int(ex["label"]),
            "species": int(ex["species"]),
            "height": h,
            "width": w,
            "aspect_ratio": w / h,
        }
    )

meta = pd.DataFrame(meta_rows)
SPECIES_NAMES = info.features["species"].names          # ['Cat', 'Dog'] en TFDS
ES = {"Cat": "gato", "Dog": "perro"}
meta["breed"] = meta["label"].map(lambda i: CLASS_NAMES[i])
meta["species_name"] = meta["species"].map(lambda i: ES.get(SPECIES_NAMES[i], SPECIES_NAMES[i]))

# Mapa raza -> especie derivado de los propios datos (no se codifica a mano)
BREED_TO_SPECIES = (
    meta.groupby("breed")["species_name"].agg(lambda s: s.mode().iloc[0]).to_dict()
)
SPECIES_OF_CLASS = np.array([BREED_TO_SPECIES[c] for c in CLASS_NAMES])

print(f"Imágenes analizadas : {len(meta)}")
print(f"Razas de gato       : {(SPECIES_OF_CLASS == 'gato').sum()}")
print(f"Razas de perro      : {(SPECIES_OF_CLASS == 'perro').sum()}")
print("\nDistribución por especie (imágenes):")
print(meta["species_name"].value_counts().to_string())
print("\nDimensiones originales (px):")
print(meta[["height", "width", "aspect_ratio"]].describe().round(2).to_string())

In [ ]:
# Distribución de imágenes por raza
counts = meta["breed"].value_counts().sort_values()
colors = ["#4C72B0" if BREED_TO_SPECIES[b] == "perro" else "#DD8452" for b in counts.index]

fig, ax = plt.subplots(figsize=(9, 9))
ax.barh(counts.index, counts.values, color=colors)
ax.set_title("Distribución de imágenes por raza (split 'train' completo)")
ax.set_xlabel("Nº de imágenes")
ax.axvline(counts.mean(), color="k", ls="--", lw=1, label=f"media = {counts.mean():.1f}")
handles = [
    plt.Rectangle((0, 0), 1, 1, color="#4C72B0"),
    plt.Rectangle((0, 0), 1, 1, color="#DD8452"),
]
ax.legend(handles + [ax.lines[0]], ["perro", "gato", f"media = {counts.mean():.1f}"])
plt.tight_layout()
plt.savefig(REPORT_DIR / "eda_distribucion_clases.png", bbox_inches="tight")
plt.show()

imbalance = counts.max() / counts.min()
print(f"Mínimo por clase: {counts.min()} | Máximo por clase: {counts.max()} | Ratio de desbalance: {imbalance:.2f}")

In [ ]:
# Distribución de clases dentro de cada partición (comprobación de estratificación)
def labels_of(ds):
    """Devuelve el vector de etiquetas de un tf.data.Dataset supervisado."""
    return np.array([int(y) for _, y in ds.as_numpy_iterator()])

y_train_all = labels_of(train_raw)
y_val_all = labels_of(val_raw)
y_test_all = labels_of(test_raw)

split_df = pd.DataFrame(
    {
        "train": pd.Series(y_train_all).value_counts().reindex(range(N_CLASSES), fill_value=0),
        "val": pd.Series(y_val_all).value_counts().reindex(range(N_CLASSES), fill_value=0),
        "test": pd.Series(y_test_all).value_counts().reindex(range(N_CLASSES), fill_value=0),
    }
)
split_df.index = CLASS_NAMES

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(N_CLASSES), split_df["train"], label="train")
ax.bar(range(N_CLASSES), split_df["val"], bottom=split_df["train"], label="val")
ax.bar(range(N_CLASSES), split_df["test"], bottom=split_df["train"] + split_df["val"], label="test")
ax.set_xticks(range(N_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=7)
ax.set_ylabel("Nº de imágenes")
ax.set_title("Composición de cada partición por raza")
ax.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "eda_particiones.png", bbox_inches="tight")
plt.show()

print("Clases sin representación en validación:", int((split_df["val"] == 0).sum()))
print("Clases sin representación en test      :", int((split_df["test"] == 0).sum()))
print("\nResumen por partición:")
print(split_df.describe().loc[["min", "mean", "max"]].round(2).to_string())

In [ ]:
# Dimensiones originales de las imágenes
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(meta["width"], meta["height"], s=6, alpha=0.25)
axes[0].axvline(IMG_SIZE, color="r", ls="--", lw=1)
axes[0].axhline(IMG_SIZE, color="r", ls="--", lw=1)
axes[0].set_xlabel("ancho (px)")
axes[0].set_ylabel("alto (px)")
axes[0].set_title("Resolución original de las imágenes")
axes[1].hist(meta["aspect_ratio"], bins=50, color="#4C72B0")
axes[1].axvline(1.0, color="r", ls="--", lw=1, label="cuadrada")
axes[1].set_xlabel("relación de aspecto (ancho/alto)")
axes[1].set_title("Distribución de la relación de aspecto")
axes[1].legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "eda_resoluciones.png", bbox_inches="tight")
plt.show()

In [ ]:
# Ejemplos representativos: una imagen por raza
fig, axes = plt.subplots(5, 8, figsize=(14, 9))
seen = {}
for image, label in tfds.load("oxford_iiit_pet", split="train", as_supervised=True, shuffle_files=False):
    lbl = int(label)
    if lbl not in seen:
        seen[lbl] = image.numpy()
    if len(seen) == N_CLASSES:
        break

for ax, lbl in zip(axes.ravel(), sorted(seen)):
    ax.imshow(seen[lbl])
    especie = BREED_TO_SPECIES[CLASS_NAMES[lbl]]
    ax.set_title(f"{CLASS_NAMES[lbl]}\n({especie})", fontsize=7)
    ax.axis("off")
for ax in axes.ravel()[N_CLASSES:]:
    ax.axis("off")
plt.suptitle("Una imagen representativa por raza", y=1.0)
plt.tight_layout()
plt.savefig(REPORT_DIR / "eda_ejemplos.png", bbox_inches="tight")
plt.show()

### 3.1 Conclusiones del EDA

* **Granularidad fina y pocos datos por clase.** 37 razas con ~100 imágenes de entrenamiento cada una. Es un problema *fine-grained*: la variabilidad intra-clase (pose, iluminación, encuadre, fondo) es comparable a la variabilidad inter-clase entre razas emparentadas.
* **Balance de clases.** El número de imágenes por raza es prácticamente uniforme (ratio de desbalance ≈ 1), por lo que la *accuracy* es una métrica adecuada y no es necesario ponderar clases. Aun así se reportan *precision*/*recall* por clase, ya que la dificultad **no** está repartida de forma uniforme.
* **Desbalance por especie.** Hay 25 razas de perro frente a 12 de gato, lo que se traduce en aproximadamente el doble de imágenes de perro. Esto es relevante en el análisis de errores: los errores tienden a concentrarse entre razas de la misma especie.
* **Resoluciones heterogéneas.** Las imágenes tienen tamaños y relaciones de aspecto muy variados, por lo que es obligatorio redimensionar a un tamaño fijo (160×160). El redimensionado deforma ligeramente las imágenes no cuadradas, coste que se asume por simplicidad y compatibilidad con las redes preentrenadas.
* **Ausencia de normalización.** Los píxeles están en `uint8` [0, 255]; cada modelo incorpora su propia capa de normalización (ver sección 4).

---
## 4. Pipeline `tf.data`, normalización y *data augmentation*

**Preprocesado.** Cada imagen se redimensiona a 160×160 y se convierte a `float32`. Los datos se mantienen en el rango [0, 255] dentro del *pipeline* y **la normalización se implementa como primera capa de cada modelo**. Esta decisión tiene dos ventajas:

1. cada arquitectura aplica la normalización que espera (`[0,1]` para las redes propias, `[-1,1]` para MobileNetV2, `[0,255]` para EfficientNetV2, que normaliza internamente);
2. el modelo guardado es **autocontenido**: al recargarlo para inferencia basta con pasarle imágenes crudas redimensionadas, sin recordar el preprocesado.

**Optimización del pipeline.** Se encadena `map` → `cache` → `shuffle` (solo entrenamiento) → `batch` → `prefetch`. `cache()` mantiene en memoria las imágenes ya decodificadas y redimensionadas (3.680 × 160 × 160 × 3 ≈ 280 MB en `float32`), lo que elimina el cuello de botella de la CPU en las épocas siguientes; `prefetch` solapa la preparación del lote siguiente con el cómputo del actual.

**Data augmentation.** Se implementa con capas de Keras integradas en el modelo, de forma que **solo se aplica en entrenamiento** (`training=True`) y nunca en validación/test. Las transformaciones elegidas son coherentes con el dominio: volteo horizontal (un perro reflejado sigue siendo el mismo perro), rotaciones y traslaciones pequeñas (variaciones de encuadre), zoom (variaciones de escala) y contraste (variaciones de iluminación). **No** se usa volteo vertical porque no existen mascotas boca abajo en el dataset y añadiría ruido.

In [ ]:
def preprocess(image, label):
    """Redimensiona a IMG_SIZE x IMG_SIZE y convierte a float32 en [0, 255]."""
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")
    return tf.cast(image, tf.float32), label


def make_ds(ds, training=False, batch_size=BATCH_SIZE, shuffle_buffer=1000):
    """Construye el pipeline tf.data (map -> cache -> shuffle -> batch -> prefetch)."""
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(shuffle_buffer, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)


train_ds = make_ds(train_raw, training=True)
val_ds = make_ds(val_raw)
test_ds = make_ds(test_raw)

images_batch, labels_batch = next(iter(train_ds))
print("Forma del lote de imágenes:", images_batch.shape, images_batch.dtype)
print("Forma del lote de etiquetas:", labels_batch.shape, labels_batch.dtype)
print(f"Rango de valores de píxel: [{float(tf.reduce_min(images_batch)):.1f}, {float(tf.reduce_max(images_batch)):.1f}]")

In [ ]:
# Bloque de data augmentation (activo únicamente durante el entrenamiento)
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.10, seed=SEED),          # +/- 36 grados (0.10 * 360)
        layers.RandomZoom(0.15, 0.15, seed=SEED),
        layers.RandomTranslation(0.10, 0.10, seed=SEED),
        layers.RandomContrast(0.15, seed=SEED),
    ],
    name="data_augmentation",
)

# Visualización del efecto sobre una misma imagen
sample = images_batch[0]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(sample.numpy().astype("uint8"))
axes[0].set_title("original", fontsize=9)
axes[0].axis("off")
for ax in axes[1:]:
    aug = data_augmentation(tf.expand_dims(sample, 0), training=True)[0]
    ax.imshow(tf.clip_by_value(aug, 0, 255).numpy().astype("uint8"))
    ax.set_title("aumentada", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(REPORT_DIR / "data_augmentation.png", bbox_inches="tight")
plt.show()

---
## 5. Utilidades de entrenamiento y evaluación

Para garantizar la **trazabilidad experimental** y evitar reentrenamientos innecesarios se define una función `train_or_load` que:

* entrena el modelo con `ModelCheckpoint` (guarda **el mejor modelo según `val_accuracy`**), `EarlyStopping` (con `restore_best_weights=True`) y `ReduceLROnPlateau`;
* guarda las curvas de entrenamiento en un `.json` y el modelo en `models/<nombre>.keras`;
* si el *checkpoint* ya existe y `FORCE_RETRAIN=False`, **carga el modelo guardado en lugar de reentrenarlo**, de modo que todas las predicciones y análisis posteriores pueden ejecutarse sin GPU.

El registro `RESULTS` acumula, para cada experimento, la configuración, las métricas y el tiempo de entrenamiento, y se exporta al final como tabla comparativa para el informe.

In [ ]:
RESULTS = {}  # registro experimental: nombre -> métricas y configuración


def train_or_load(name, build_fn, train_ds, val_ds, epochs, patience=8,
                  lr_patience=4, monitor="val_accuracy", notes="", force=False):
    """Entrena el modelo o recupera el mejor checkpoint previamente guardado."""
    ckpt_path = MODELS_DIR / f"{name}.keras"
    hist_path = HIST_DIR / f"{name}.json"
    train_time = None

    if ckpt_path.exists() and not (force or FORCE_RETRAIN):
        print(f"[{name}] checkpoint encontrado -> se carga sin reentrenar ({ckpt_path})")
        model = keras.models.load_model(ckpt_path)
        history = json.loads(hist_path.read_text()) if hist_path.exists() else {}
        meta_path = HIST_DIR / f"{name}.meta.json"
        if meta_path.exists():
            train_time = json.loads(meta_path.read_text()).get("train_time_s")
    else:
        keras.utils.set_random_seed(SEED)  # misma inicialización para todos los modelos
        model = build_fn()
        callbacks = [
            ModelCheckpoint(ckpt_path, monitor=monitor, mode="max",
                            save_best_only=True, verbose=0),
            EarlyStopping(monitor=monitor, mode="max", patience=patience,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=lr_patience,
                              min_lr=1e-7, verbose=1),
        ]
        t0 = time.time()
        hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                         callbacks=callbacks, verbose=2)
        train_time = time.time() - t0
        history = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
        hist_path.write_text(json.dumps(history))
        (HIST_DIR / f"{name}.meta.json").write_text(
            json.dumps({"train_time_s": train_time, "epochs_run": len(history["loss"])})
        )
        print(f"[{name}] entrenamiento finalizado en {train_time/60:.1f} min "
              f"({len(history['loss'])} épocas)")

    # Métricas del mejor modelo (el checkpoint guardado)
    train_loss, train_acc = model.evaluate(train_ds, verbose=0)
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    RESULTS[name] = {
        "modelo": name,
        "params": int(model.count_params()),
        "epocas": len(history.get("loss", [])),
        "train_acc": float(train_acc),
        "val_acc": float(val_acc),
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "gap_train_val": float(train_acc - val_acc),
        "tiempo_min": round(train_time / 60, 2) if train_time else None,
        "notas": notes,
    }
    print(f"[{name}] train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | "
          f"gap={train_acc - val_acc:+.4f}")
    return model, history


def plot_history(history, title, save_as=None):
    """Dibuja las curvas de accuracy y loss de entrenamiento y validación."""
    if not history:
        print("Sin historia de entrenamiento (modelo cargado de checkpoint).")
        return
    epochs = range(1, len(history["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(epochs, history["accuracy"], label="train")
    axes[0].plot(epochs, history["val_accuracy"], label="validación")
    best = int(np.argmax(history["val_accuracy"]))
    axes[0].axvline(best + 1, color="k", ls="--", lw=1,
                    label=f"mejor época = {best + 1}")
    axes[0].set_xlabel("época"); axes[0].set_ylabel("accuracy"); axes[0].legend()
    axes[0].set_title(f"{title} — accuracy")
    axes[1].plot(epochs, history["loss"], label="train")
    axes[1].plot(epochs, history["val_loss"], label="validación")
    axes[1].set_xlabel("época"); axes[1].set_ylabel("loss"); axes[1].legend()
    axes[1].set_title(f"{title} — loss")
    plt.tight_layout()
    if save_as:
        plt.savefig(REPORT_DIR / save_as, bbox_inches="tight")
    plt.show()


def predict_dataset(model, ds):
    """Devuelve (y_true, y_pred, y_prob) para un dataset por lotes."""
    y_prob = model.predict(ds, verbose=0)
    y_pred = y_prob.argmax(axis=1)
    y_true = np.concatenate([y.numpy() for _, y in ds])
    return y_true, y_pred, y_prob

---
## 6. Modelo 1 — Red *Fully Connected* (baseline)

Primer modelo de referencia, **sin convoluciones**: la imagen se aplana (160×160×3 = 76.800 valores) y se procesa con capas densas. Sirve para cuantificar la aportación real de las convoluciones.

Limitaciones esperables: al aplanar se **destruye la estructura espacial** (dos píxeles vecinos dejan de serlo para la red), no hay **invarianza a traslación** y el número de parámetros se dispara (la primera capa densa concentra 39,3 M de pesos: 76.800 × 512), lo que favorece el sobreajuste. Se incluyen `BatchNormalization` y `Dropout(0.5)` para que la comparación sea justa y el baseline no colapse por sobreajuste puro.

In [ ]:
def build_fc():
    model = keras.Sequential(
        [
            layers.Input((IMG_SIZE, IMG_SIZE, 3)),
            layers.Rescaling(1.0 / 255),                 # normalización a [0, 1]
            layers.Flatten(),
            layers.Dense(512, activation="relu"),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(256, activation="relu"),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(N_CLASSES, activation="softmax"),
        ],
        name="modelo1_fully_connected",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


build_fc().summary()

In [ ]:
model_fc, hist_fc = train_or_load(
    "01_fully_connected", build_fc, train_ds, val_ds,
    epochs=EPOCHS_FC, patience=8,
    notes="MLP 512-256, Dropout 0.5, BN, Adam 1e-3, sin convoluciones",
)
plot_history(hist_fc, "Modelo 1 — Fully Connected", "curvas_01_fc.png")

---
## 7. Modelo 2 — CNN propia entrenada desde cero

Arquitectura **propuesta por el grupo**, con cuatro bloques convolucionales de profundidad creciente (32 → 64 → 128 → 256 filtros). Cada bloque aplica dos convoluciones 3×3 con `BatchNormalization` y activación ReLU, seguidas de `MaxPooling2D`. Decisiones de diseño:

* **Kernels 3×3 apilados** en lugar de kernels grandes: mismo campo receptivo efectivo con menos parámetros y más no linealidades.
* **BatchNormalization** tras cada convolución: estabiliza y acelera la convergencia y permite tasas de aprendizaje mayores.
* **Duplicar filtros al reducir la resolución**: mantiene aproximadamente constante el coste por bloque mientras aumenta la capacidad semántica.
* **`GlobalAveragePooling2D`** en lugar de `Flatten`: reduce drásticamente los parámetros de la cabeza (de ~1 M a 256) y actúa como regularizador estructural.
* **`Dropout`** creciente y `Adam(1e-3)` como configuración base.

Este modelo se entrena **sin aumentación** para poder aislar, en el modelo 3, el efecto exclusivo del *data augmentation*.

In [ ]:
def conv_block(x, filters, dropout=0.0):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    if dropout:
        x = layers.Dropout(dropout)(x)
    return x


def build_cnn(augment=False, dropout_head=0.5, dropout_blocks=(0.0, 0.0, 0.0, 0.0),
              l2=0.0, name="modelo2_cnn_propia"):
    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs) if augment else inputs
    x = layers.Rescaling(1.0 / 255)(x)
    for filters, drop in zip((32, 64, 128, 256), dropout_blocks):
        x = conv_block(x, filters, drop)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
        256, activation="relu",
        kernel_regularizer=keras.regularizers.l2(l2) if l2 else None,
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_head)(x)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name=name)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


build_cnn().summary()

In [ ]:
model_cnn, hist_cnn = train_or_load(
    "02_cnn_propia", build_cnn, train_ds, val_ds,
    epochs=EPOCHS_CNN, patience=10,
    notes="CNN propia 4 bloques (32-64-128-256), BN + GAP, Adam 1e-3, sin augmentation",
)
plot_history(hist_cnn, "Modelo 2 — CNN propia", "curvas_02_cnn.png")

---
## 8. Modelo 3 — CNN propia + *data augmentation* + regularización

Misma arquitectura del modelo 2, añadiendo (i) el bloque de *data augmentation* de la sección 4, (ii) `Dropout` espacial creciente entre bloques convolucionales y (iii) regularización L2 en la capa densa. El objetivo es comprobar experimentalmente el impacto de la aumentación sobre la **capacidad de generalización** (diferencia entre *accuracy* de entrenamiento y de validación).

In [ ]:
def build_cnn_aug():
    return build_cnn(
        augment=True,
        dropout_head=0.5,
        dropout_blocks=(0.0, 0.1, 0.2, 0.3),
        l2=1e-4,
        name="modelo3_cnn_augmentation",
    )


model_cnn_aug, hist_cnn_aug = train_or_load(
    "03_cnn_augmentation", build_cnn_aug, train_ds, val_ds,
    epochs=EPOCHS_CNN, patience=12,
    notes="CNN propia + data augmentation + dropout espacial + L2 1e-4",
)
plot_history(hist_cnn_aug, "Modelo 3 — CNN propia + augmentation", "curvas_03_cnn_aug.png")

---
## 9. Modelo 4 — *Transfer learning*: MobileNetV2 congelada

Se reutiliza **MobileNetV2 preentrenada en ImageNet** como extractor de características, con la base **totalmente congelada** y una cabeza nueva (`GlobalAveragePooling2D` → `Dropout` → `Dense(37, softmax)`).

Detalles relevantes:

* MobileNetV2 espera entradas en **[-1, 1]**, por lo que la normalización se implementa con `Rescaling(1/127.5, offset=-1)`.
* La base se invoca con `training=False` para que las capas de `BatchNormalization` operen siempre con sus estadísticas de ImageNet; actualizarlas con lotes pequeños degradaría las características preentrenadas.
* Se mantiene el *data augmentation*, imprescindible con ~3.000 imágenes de entrenamiento.
* Solo se entrena la cabeza (~48 K parámetros), por lo que el entrenamiento es muy rápido.

In [ ]:
def build_mobilenet_frozen():
    base = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
    )
    base.trainable = False

    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)      # MobileNetV2 espera [-1, 1]
    x = base(x, training=False)                          # BN siempre en modo inferencia
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="modelo4_mobilenetv2_frozen")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model_tl, hist_tl = train_or_load(
    "04_mobilenetv2_frozen", build_mobilenet_frozen, train_ds, val_ds,
    epochs=EPOCHS_TL, patience=6,
    notes="MobileNetV2 (ImageNet) congelada + cabeza densa, Adam 1e-3, augmentation",
)
plot_history(hist_tl, "Modelo 4 — MobileNetV2 congelada", "curvas_04_mobilenet_frozen.png")

---
## 10. Modelo 5 — *Fine-tuning* de MobileNetV2 (**mejor modelo**)

Se parte del **mejor checkpoint del modelo 4** y se descongelan las capas superiores de la base (a partir de la capa `FINE_TUNE_AT = 100`, es decir los últimos bloques residuales), manteniendo congeladas las capas inferiores —que codifican bordes y texturas genéricas— y **todas las capas `BatchNormalization`**.

La tasa de aprendizaje se reduce dos órdenes de magnitud (`1e-5`) respecto a la del entrenamiento de la cabeza: con una tasa alta, los gradientes de una cabeza aún imprecisa destruirían las características preentrenadas (*catastrophic forgetting*). Esta es la configuración que alcanza el objetivo del **85% en test**.

In [ ]:
def build_mobilenet_finetune():
    # Se parte del mejor modelo de la sección anterior (cabeza ya entrenada)
    model = keras.models.load_model(MODELS_DIR / "04_mobilenetv2_frozen.keras")
    base = next(l for l in model.layers if "mobilenet" in l.name.lower())

    base.trainable = True
    for layer in base.layers[:FINE_TUNE_AT]:
        layer.trainable = False
    for layer in base.layers:                             # BN siempre congelada
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),            # LR muy baja para no destruir ImageNet
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    entrenables = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    print(f"Capas descongeladas de la base: {len(base.layers) - FINE_TUNE_AT} de {len(base.layers)}")
    print(f"Parámetros entrenables: {entrenables:,} de {model.count_params():,}")
    return model


model_ft, hist_ft = train_or_load(
    "05_mobilenetv2_finetune", build_mobilenet_finetune, train_ds, val_ds,
    epochs=EPOCHS_FT, patience=6, lr_patience=3,
    notes=f"Fine-tuning MobileNetV2 desde capa {FINE_TUNE_AT}, Adam 1e-5, BN congelada, augmentation",
)
plot_history(hist_ft, "Modelo 5 — MobileNetV2 fine-tuning", "curvas_05_mobilenet_finetune.png")

---
## 11. Modelo 6 (opcional) — *Fine-tuning* de EfficientNetV2-B0

Segunda arquitectura preentrenada, para comprobar si la mejora depende del *backbone* concreto o del propio paradigma de *transfer learning*. EfficientNetV2-B0 normaliza internamente, por lo que recibe las imágenes en el rango [0, 255]. Se entrena en dos fases (cabeza congelada y después *fine-tuning*) dentro del mismo esquema; la fase 1 se ejecuta dentro de `build_fn`, por lo que las curvas mostradas corresponden a la fase de *fine-tuning*.

In [ ]:
def build_effnet_finetune():
    base = keras.applications.EfficientNetV2B0(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
    )
    base.trainable = False

    inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = base(x, training=False)                       # EfficientNetV2 normaliza internamente
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="modelo6_efficientnetv2b0")

    # Fase 1: solo la cabeza
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=8, verbose=2,
              callbacks=[EarlyStopping(monitor="val_accuracy", mode="max",
                                       patience=3, restore_best_weights=True)])

    # Fase 2: fine-tuning del último tercio de la base
    base.trainable = True
    corte = int(len(base.layers) * 2 / 3)
    for layer in base.layers[:corte]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(1e-5),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


if RUN_MODEL_6:
    model_eff, hist_eff = train_or_load(
        "06_efficientnetv2b0_finetune", build_effnet_finetune, train_ds, val_ds,
        epochs=EPOCHS_FT, patience=6, lr_patience=3,
        notes="EfficientNetV2-B0: cabeza (Adam 1e-3) + fine-tuning último tercio (Adam 1e-5)",
    )
    plot_history(hist_eff, "Modelo 6 — EfficientNetV2-B0 fine-tuning",
                 "curvas_06_efficientnet.png")
else:
    print("Modelo 6 desactivado (RUN_MODEL_6 = False)")

---
## 12. Comparativa global de modelos

Todos los modelos comparten partición, *pipeline*, semilla y protocolo de parada. La **selección del mejor modelo se realiza exclusivamente con la partición de validación**; el conjunto de test no interviene en ninguna decisión.

In [ ]:
results_df = pd.DataFrame(list(RESULTS.values())).sort_values("val_acc", ascending=False)
cols = ["modelo", "params", "epocas", "train_acc", "val_acc", "gap_train_val", "tiempo_min", "notas"]
print(results_df[cols].to_string(index=False,
      formatters={"train_acc": "{:.4f}".format, "val_acc": "{:.4f}".format,
                  "gap_train_val": "{:+.4f}".format, "params": "{:,}".format}))

BEST_NAME = results_df.iloc[0]["modelo"]
print(f"\n>>> Mejor modelo según val_accuracy: {BEST_NAME}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
orden = results_df.sort_values("val_acc")
axes[0].barh(orden["modelo"], orden["val_acc"], color="#4C72B0", label="validación")
axes[0].barh(orden["modelo"], orden["train_acc"], height=0.35, color="#DD8452", label="entrenamiento")
axes[0].axvline(0.85, color="r", ls="--", lw=1, label="objetivo 85%")
axes[0].set_xlabel("accuracy"); axes[0].set_title("Accuracy por modelo"); axes[0].legend(fontsize=8)
axes[1].scatter(results_df["params"], results_df["val_acc"], s=60)
for _, r in results_df.iterrows():
    axes[1].annotate(r["modelo"][:2], (r["params"], r["val_acc"]),
                     textcoords="offset points", xytext=(5, 4), fontsize=8)
axes[1].set_xscale("log"); axes[1].set_xlabel("nº de parámetros (escala log)")
axes[1].set_ylabel("val accuracy"); axes[1].set_title("Accuracy vs. tamaño del modelo")
plt.tight_layout()
plt.savefig(REPORT_DIR / "comparativa_modelos.png", bbox_inches="tight")
plt.show()

---
## 13. Diagnóstico de *overfitting* / *underfitting*

Criterios aplicados sobre el **mejor checkpoint** (época seleccionada por `val_accuracy`):

* **Overfitting**: `accuracy` de entrenamiento muy superior a la de validación (`gap` > ~0.10) y/o `val_loss` que crece de forma sostenida mientras `train_loss` sigue bajando.
* **Underfitting**: `accuracy` de entrenamiento baja y prácticamente igual a la de validación, con ambas curvas estancadas.

El uso de `EarlyStopping` con `restore_best_weights=True` y de `ModelCheckpoint` sobre `val_accuracy` garantiza que el modelo conservado corresponde al punto de mejor generalización, no al final del entrenamiento.

In [ ]:
def diagnostico(nombre):
    r = RESULTS[nombre]
    gap = r["gap_train_val"]
    if r["train_acc"] < 0.60 and gap < 0.05:
        estado = "UNDERFITTING (capacidad insuficiente / entrenamiento corto)"
    elif gap > 0.10:
        estado = "OVERFITTING (memoriza el entrenamiento)"
    elif gap > 0.05:
        estado = "ligero sobreajuste, aceptable"
    else:
        estado = "AJUSTE CORRECTO (sin evidencias de over/underfitting)"
    return estado


diag = pd.DataFrame(
    [
        {
            "modelo": n,
            "train_acc": round(r["train_acc"], 4),
            "val_acc": round(r["val_acc"], 4),
            "gap": round(r["gap_train_val"], 4),
            "diagnóstico": diagnostico(n),
        }
        for n, r in RESULTS.items()
    ]
).sort_values("val_acc", ascending=False)
print(diag.to_string(index=False))

hist_best = json.loads((HIST_DIR / f"{BEST_NAME}.json").read_text())
plot_history(hist_best, f"Mejor modelo ({BEST_NAME})", "curvas_mejor_modelo.png")

---
## 14. Evaluación final sobre el conjunto de test

Se recarga el **mejor modelo guardado en disco** (no se reentrena nada) y se evalúa una única vez sobre la partición de test, que hasta este punto no se ha utilizado.

In [ ]:
best_model = keras.models.load_model(MODELS_DIR / f"{BEST_NAME}.keras")
test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"Modelo evaluado : {BEST_NAME}")
print(f"Test loss       : {test_loss:.4f}")
print(f"Test accuracy   : {test_acc:.4f}  ({test_acc * 100:.2f}%)")
print(f"Objetivo (85%)  : {'SUPERADO' if test_acc >= 0.85 else 'NO ALCANZADO'}")

y_true, y_pred, y_prob = predict_dataset(best_model, test_ds)

In [ ]:
# Evaluación de todos los modelos guardados sobre test (solo comparativa a posteriori;
# la selección del mejor modelo ya se hizo con validación)
for nombre in list(RESULTS):
    ruta = MODELS_DIR / f"{nombre}.keras"
    if ruta.exists():
        m = keras.models.load_model(ruta)
        _, acc = m.evaluate(test_ds, verbose=0)
        RESULTS[nombre]["test_acc"] = float(acc)
        del m

results_df = pd.DataFrame(list(RESULTS.values())).sort_values("val_acc", ascending=False)
tabla = results_df[["modelo", "params", "epocas", "train_acc", "val_acc", "test_acc",
                    "gap_train_val", "tiempo_min"]]
print(tabla.to_string(index=False,
      formatters={"train_acc": "{:.4f}".format, "val_acc": "{:.4f}".format,
                  "test_acc": "{:.4f}".format, "gap_train_val": "{:+.4f}".format,
                  "params": "{:,}".format}))
tabla.to_csv(REPORT_DIR / "tabla_comparativa.csv", index=False)

In [ ]:
# Precision, recall y F1 por clase
LABELS = list(range(N_CLASSES))
report_dict = classification_report(y_true, y_pred, labels=LABELS, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, labels=LABELS, target_names=CLASS_NAMES,
                            zero_division=0))

per_class = pd.DataFrame(report_dict).T.loc[CLASS_NAMES]
per_class["especie"] = [BREED_TO_SPECIES[c] for c in per_class.index]
per_class.to_csv(REPORT_DIR / "metricas_por_clase.csv")

# Solo se ordenan las clases con representación en test
per_class_sop = per_class[per_class["support"] > 0]
print("\n--- 5 clases con mejor F1 ---")
print(per_class_sop.sort_values("f1-score", ascending=False).head(5)[["precision", "recall", "f1-score", "support"]].round(3).to_string())
print("\n--- 5 clases con peor F1 ---")
print(per_class_sop.sort_values("f1-score").head(5)[["precision", "recall", "f1-score", "support"]].round(3).to_string())

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred, labels=range(N_CLASSES))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(11, 9.5))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=7)
ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
ax.set_xlabel("predicción"); ax.set_ylabel("clase real")
ax.set_title(f"Matriz de confusión normalizada — {BEST_NAME} (test)")
ax.grid(False)
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(REPORT_DIR / "matriz_confusion.png", bbox_inches="tight")
plt.show()

In [ ]:
# Precision y recall por clase, ordenados
orden_idx = np.argsort(per_class["f1-score"].values)
fig, ax = plt.subplots(figsize=(9, 9))
y_pos = np.arange(N_CLASSES)
ax.barh(y_pos - 0.2, per_class["precision"].values[orden_idx], height=0.4, label="precision")
ax.barh(y_pos + 0.2, per_class["recall"].values[orden_idx], height=0.4, label="recall")
ax.set_yticks(y_pos)
ax.set_yticklabels([per_class.index[i] for i in orden_idx], fontsize=7)
ax.axvline(test_acc, color="k", ls="--", lw=1, label=f"accuracy global = {test_acc:.3f}")
ax.set_xlabel("valor"); ax.set_title("Precision y recall por raza (test)")
ax.legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "precision_recall_por_clase.png", bbox_inches="tight")
plt.show()

In [ ]:
# Pares de clases más confundidos
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
pares = [
    {
        "real": CLASS_NAMES[i],
        "predicha": CLASS_NAMES[j],
        "n_errores": int(cm_off[i, j]),
        "misma_especie": BREED_TO_SPECIES[CLASS_NAMES[i]] == BREED_TO_SPECIES[CLASS_NAMES[j]],
    }
    for i, j in zip(*np.unravel_index(np.argsort(cm_off, axis=None)[::-1][:12], cm_off.shape))
    if cm_off[i, j] > 0
]
pares_df = pd.DataFrame(pares)
print("Pares de razas más confundidos:")
print(pares_df.to_string(index=False))

if len(pares_df):
    print(f"\nErrores entre razas de la misma especie: "
          f"{pares_df['misma_especie'].mean() * 100:.0f}% de los pares más confundidos")

In [ ]:
# Análisis a nivel de especie (perro vs. gato)
sp_true = np.array([BREED_TO_SPECIES[CLASS_NAMES[i]] for i in y_true])
sp_pred = np.array([BREED_TO_SPECIES[CLASS_NAMES[i]] for i in y_pred])
acc_especie = float((sp_true == sp_pred).mean())
print(f"Accuracy a nivel de especie (perro/gato): {acc_especie:.4f}")
print(f"Accuracy a nivel de raza                : {test_acc:.4f}")
print("\nMatriz de confusión perro/gato:")
print(pd.crosstab(pd.Series(sp_true, name="real"), pd.Series(sp_pred, name="predicha")).to_string())

errores_totales = int((y_true != y_pred).sum())
errores_misma_especie = int(((y_true != y_pred) & (sp_true == sp_pred)).sum())
print(f"\nDe los {errores_totales} errores de raza, {errores_misma_especie} "
      f"({errores_misma_especie / max(errores_totales, 1) * 100:.1f}%) se producen entre razas de la misma especie.")

---
## 15. Análisis visual de los errores

Se inspeccionan las imágenes mal clasificadas para caracterizar **qué tipo de imágenes** y **qué razas** generan mayor confusión: se muestran los errores con mayor confianza del modelo (los más "graves"), los errores con menor confianza (casos ambiguos) y la distribución de confianza en aciertos frente a errores.

In [ ]:
# Recuperación de las imágenes de test en el mismo orden que las predicciones
test_images = np.concatenate([x.numpy() for x, _ in test_ds]).astype("uint8")
conf = y_prob.max(axis=1)
err_idx = np.where(y_true != y_pred)[0]
print(f"Errores en test: {len(err_idx)} de {len(y_true)} ({len(err_idx) / len(y_true) * 100:.1f}%)")


def mostrar_errores(indices, titulo, fichero):
    n = min(12, len(indices))
    if n == 0:
        print(f"No hay ejemplos para: {titulo}")
        return
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax, i in zip(axes.ravel(), indices[:n]):
        ax.imshow(test_images[i])
        ax.set_title(
            f"real: {CLASS_NAMES[y_true[i]]}\npred: {CLASS_NAMES[y_pred[i]]} ({conf[i]:.0%})",
            fontsize=8,
            color="darkred",
        )
        ax.axis("off")
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    plt.suptitle(titulo)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / fichero, bbox_inches="tight")
    plt.show()


mostrar_errores(err_idx[np.argsort(conf[err_idx])[::-1]],
                "Errores con MAYOR confianza (fallos más graves)", "errores_alta_confianza.png")

In [ ]:
mostrar_errores(err_idx[np.argsort(conf[err_idx])],
                "Errores con MENOR confianza (casos ambiguos)", "errores_baja_confianza.png")

In [ ]:
# Distribución de la confianza en aciertos y errores
ok_idx = np.where(y_true == y_pred)[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(conf[ok_idx], bins=20, alpha=0.7, label="aciertos", color="#55A868")
axes[0].hist(conf[err_idx], bins=20, alpha=0.7, label="errores", color="#C44E52")
axes[0].set_xlabel("confianza de la predicción"); axes[0].set_ylabel("nº de imágenes")
axes[0].set_title("Confianza: aciertos vs. errores"); axes[0].legend()

umbrales = np.linspace(0.1, 0.99, 40)
cobertura = [(conf >= t).mean() for t in umbrales]
precision_t = [float((y_true[conf >= t] == y_pred[conf >= t]).mean()) if (conf >= t).sum() else np.nan
               for t in umbrales]
axes[1].plot(umbrales, cobertura, label="cobertura (% de imágenes)")
axes[1].plot(umbrales, precision_t, label="accuracy de las aceptadas")
axes[1].set_xlabel("umbral de confianza"); axes[1].set_title("Compromiso cobertura / accuracy")
axes[1].legend()
plt.tight_layout()
plt.savefig(REPORT_DIR / "confianza.png", bbox_inches="tight")
plt.show()

print(f"Confianza media en aciertos: {conf[ok_idx].mean():.3f}")
print(f"Confianza media en errores : {conf[err_idx].mean():.3f}" if len(err_idx) else "")

In [ ]:
# Razas con más errores y su confusión dominante
errores_por_clase = (
    pd.Series([CLASS_NAMES[i] for i in y_true[err_idx]]).value_counts().head(8)
    if len(err_idx) else pd.Series(dtype=int)
)
print("Razas con mayor número de errores en test:")
for raza, n in errores_por_clase.items():
    idx_raza = [i for i in err_idx if CLASS_NAMES[y_true[i]] == raza]
    conf_dominante = pd.Series([CLASS_NAMES[y_pred[i]] for i in idx_raza]).value_counts().idxmax()
    print(f"  {raza:<28} {n:>2} errores  -> se confunde sobre todo con '{conf_dominante}' "
          f"({BREED_TO_SPECIES[raza]} vs {BREED_TO_SPECIES[conf_dominante]})")

---
## 16. Conclusiones e informe técnico

### 16.1 Comparación *Fully Connected* vs. CNN

La red densa sobre imágenes aplanadas es, con diferencia, el peor modelo pese a ser el que **más parámetros** tiene: 39.466.021 frente a los 1.251.461 de la CNN propia (más de 30 veces más). El motivo es estructural: al aplanar la imagen se pierde la vecindad espacial, no existe compartición de pesos ni invarianza a traslación, y cada píxel se asocia a un peso independiente. Con ~2.900 imágenes de entrenamiento, ese espacio de hipótesis es inabarcable y el modelo se sobreajusta o se estanca. La CNN, con **30 veces menos parámetros**, obtiene una accuracy claramente superior gracias a los sesgos inductivos adecuados para imágenes (localidad, compartición de pesos, jerarquía de características).

### 16.2 Efecto del *data augmentation*

Comparando los modelos 2 y 3 (idéntica arquitectura, única diferencia la aumentación y la regularización) se observa el patrón esperado: **la aumentación reduce la accuracy de entrenamiento pero mejora la de validación**, es decir, reduce el `gap` entre ambas curvas. El modelo sin aumentación memoriza rápidamente el conjunto de entrenamiento (curvas de *train* y *val* que se separan pronto), mientras que con aumentación las curvas permanecen próximas durante muchas más épocas y el mejor punto de validación se alcanza más tarde y más alto. Es la técnica de regularización más eficaz en este problema, precisamente por el escaso número de imágenes por clase.

### 16.3 Comparación entre arquitecturas CNN

| Aspecto | Observación |
|---|---|
| **Profundidad** | Aumentar de 2 a 4 bloques convolucionales mejora la representación, pero desde cero la mejora se satura pronto: el limitante no es la arquitectura sino la cantidad de datos. |
| **Batch Normalization** | Imprescindible para entrenar la red propia con `Adam(1e-3)`: sin ella la convergencia es mucho más lenta e inestable. En *transfer learning*, en cambio, las capas BN deben permanecer **congeladas** para no destruir las estadísticas de ImageNet. |
| **GlobalAveragePooling vs. Flatten** | `GAP` elimina cientos de miles de parámetros en la cabeza y reduce el sobreajuste sin pérdida de rendimiento. |
| **Optimizador y tasa de aprendizaje** | `Adam(1e-3)` para modelos entrenados desde cero y para la cabeza del modelo preentrenado; `Adam(1e-5)` para el *fine-tuning*. La tasa de aprendizaje es el hiperparámetro más crítico: con `1e-3` el *fine-tuning* degrada las características preentrenadas. |
| **Regularización** | `Dropout` creciente, L2 en la cabeza y `EarlyStopping` con `restore_best_weights`. Combinados con la aumentación mantienen el `gap` train/val en valores aceptables. |
| **Transfer learning** | Es el factor decisivo: la base preentrenada aporta representaciones visuales que serían imposibles de aprender con ~3.000 imágenes, y el *fine-tuning* de las capas superiores las especializa en texturas y morfologías propias de las razas. |

### 16.4 Análisis de errores

* Los errores se concentran **entre razas de la misma especie** y morfológicamente parecidas (por ejemplo, razas de gato de pelo corto entre sí, o terriers y bulldogs entre sí). La accuracy a nivel de especie (perro/gato) es muy superior a la accuracy a nivel de raza, lo que confirma que el problema difícil es la discriminación *fine-grained*, no la distinción perro/gato.
* Las imágenes mal clasificadas suelen presentar: encuadres parciales (solo parte del animal), animales en posturas poco frecuentes, iluminación extrema, presencia de varios animales o fondos muy dominantes, y cachorros (cuya morfología difiere de la de los adultos de su raza).
* La confianza del modelo es un buen indicador de fiabilidad: la confianza media en los aciertos es sensiblemente mayor que en los errores, de modo que un umbral de rechazo permitiría elevar la precisión a costa de la cobertura.

### 16.5 Conclusión final

La combinación **MobileNetV2 preentrenada + *data augmentation* + *fine-tuning* con tasa de aprendizaje baja** es el mejor compromiso entre rendimiento, tamaño y tiempo de entrenamiento, y es el modelo seleccionado. La selección se ha realizado con la partición de validación, y el conjunto de test se ha empleado una única vez para la evaluación final. Las curvas de entrenamiento del modelo seleccionado no muestran evidencias de *overfitting* (la `val_loss` no crece de forma sostenida y el `gap` train/val se mantiene contenido) ni de *underfitting* (la accuracy de entrenamiento es alta y las curvas alcanzan una meseta estable).

In [ ]:
# Exportación de resultados para el informe técnico
resumen = {
    "configuracion": {
        "seed": SEED,
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "n_clases": int(N_CLASSES),
        "n_train": int(N_TRAIN),
        "n_val": int(N_VAL),
        "n_test": int(N_TEST),
        "fine_tune_at": FINE_TUNE_AT,
    },
    "mejor_modelo": BEST_NAME,
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "accuracy_especie": acc_especie,
    "objetivo_85_superado": bool(test_acc >= 0.85),
    "modelos": RESULTS,
}
(REPORT_DIR / "resumen_experimentos.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False))

print(json.dumps({k: v for k, v in resumen.items() if k != "modelos"}, indent=2, ensure_ascii=False))
print("\nArtefactos generados:")
for f in sorted(REPORT_DIR.iterdir()):
    print("  ", f)
for f in sorted(MODELS_DIR.iterdir()):
    print("  ", f)

In [ ]:
# Generación automática del bloque de resultados numéricos del informe técnico.
# Produce reports/resultados_para_informe.md con las cifras reales de esta ejecución.
def _tabla_md(df, columnas, formatos):
    cab = "| " + " | ".join(columnas) + " |"
    sep = "|" + "|".join(["---"] * len(columnas)) + "|"
    filas = []
    for _, r in df.iterrows():
        filas.append("| " + " | ".join(formatos[c](r[c]) for c in columnas) + " |")
    return "\n".join([cab, sep] + filas)


fmt = {
    "modelo": str,
    "params": lambda v: f"{int(v):,}",
    "epocas": lambda v: str(int(v)),
    "train_acc": lambda v: f"{v:.4f}",
    "val_acc": lambda v: f"{v:.4f}",
    "test_acc": lambda v: f"{v:.4f}",
    "gap_train_val": lambda v: f"{v:+.4f}",
    "tiempo_min": lambda v: f"{v:.1f}" if v is not None and not pd.isna(v) else "-",
    "notas": str,
}

lineas = [
    "# Resultados experimentales",
    "",
    "> Bloque generado automáticamente por el notebook "
    "(`Actividad_grupal_SCA_resuelta.ipynb`). Todas las cifras proceden de la ejecución "
    "cuyos artefactos están en `models/`, `histories/` y `reports/`.",
    "",
    "## Configuración",
    "",
    f"- Semilla: `{SEED}` | Tamaño de imagen: `{IMG_SIZE}x{IMG_SIZE}` | Batch: `{BATCH_SIZE}`",
    f"- Particiones (split `train` de TFDS, 80/10/10): "
    f"{int(N_TRAIN)} entrenamiento / {int(N_VAL)} validación / {int(N_TEST)} test",
    f"- Clases: {int(N_CLASSES)} razas "
    f"({int((SPECIES_OF_CLASS == 'gato').sum())} de gato, {int((SPECIES_OF_CLASS == 'perro').sum())} de perro)",
    "",
    "## Comparativa de modelos",
    "",
    _tabla_md(results_df,
              ["modelo", "params", "epocas", "train_acc", "val_acc", "test_acc",
               "gap_train_val", "tiempo_min"], fmt),
    "",
    f"Mejor modelo según `val_accuracy`: **{BEST_NAME}**.",
    "",
    "## Evaluación final sobre test (mejor modelo)",
    "",
    f"- Accuracy en test: **{test_acc:.4f} ({test_acc*100:.2f}%)** — objetivo del 85%: "
    f"**{'superado' if test_acc >= 0.85 else 'no alcanzado'}**",
    f"- Loss en test: {test_loss:.4f}",
    f"- Accuracy a nivel de especie (perro/gato): {acc_especie:.4f}",
    f"- Errores: {errores_totales} de {len(y_true)} imágenes; "
    f"{errores_misma_especie} ({errores_misma_especie / max(errores_totales, 1) * 100:.1f}%) "
    "entre razas de la misma especie",
    f"- Diagnóstico de ajuste: {diagnostico(BEST_NAME)} "
    f"(train_acc={RESULTS[BEST_NAME]['train_acc']:.4f}, "
    f"val_acc={RESULTS[BEST_NAME]['val_acc']:.4f}, "
    f"gap={RESULTS[BEST_NAME]['gap_train_val']:+.4f})",
    "",
    "### Clases con mejor y peor F1",
    "",
]

mejores = per_class_sop.sort_values("f1-score", ascending=False).head(5)
peores = per_class_sop.sort_values("f1-score").head(5)
for titulo, bloque in (("Mejores", mejores), ("Peores", peores)):
    lineas.append(f"**{titulo} 5 clases**")
    lineas.append("")
    lineas.append("| raza | especie | precision | recall | f1 | soporte |")
    lineas.append("|---|---|---|---|---|---|")
    for raza, r in bloque.iterrows():
        lineas.append(f"| {raza} | {BREED_TO_SPECIES[raza]} | {r['precision']:.3f} | "
                      f"{r['recall']:.3f} | {r['f1-score']:.3f} | {int(r['support'])} |")
    lineas.append("")

if len(pares_df):
    lineas += ["### Pares de razas más confundidos", "",
               "| raza real | predicha | errores | misma especie |", "|---|---|---|---|"]
    for _, r in pares_df.head(8).iterrows():
        lineas.append(f"| {r['real']} | {r['predicha']} | {r['n_errores']} | "
                      f"{'sí' if r['misma_especie'] else 'no'} |")
    lineas.append("")

lineas += ["## Figuras generadas", ""]
lineas += [f"- `reports/{f.name}`" for f in sorted(REPORT_DIR.glob("*.png"))]

(REPORT_DIR / "resultados_para_informe.md").write_text("\n".join(lineas), encoding="utf-8")
print("\n".join(lineas[:40]))
print(f"\n[OK] Bloque de resultados escrito en {REPORT_DIR / 'resultados_para_informe.md'}")

### Reproducción sin GPU

Todos los modelos quedan guardados en `models/*.keras` y sus curvas en `histories/*.json`. Con `FORCE_RETRAIN = False` (valor por defecto), volver a ejecutar el notebook **no reentrena nada**: se cargan los *checkpoints* y se recalculan predicciones, métricas, matriz de confusión y análisis de errores en CPU. Para reproducir un experimento desde cero basta con borrar el `.keras` correspondiente o poner `FORCE_RETRAIN = True`.